In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
import random

load_dotenv()

ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset
from sklearn.metrics import f1_score

In [ ]:
# Config
config = {
    'num_labels': 188,
    'seed': 42,
    'learning_rate': 3e-5,
    'batch_size': 8,
    'epochs': 12,
    'early_stop': 5,
    'test_step': 0,
    'weight_decay': 0.01,
    'model_name': 'monologg/kobert',
}

In [ ]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("ingyoun/patent-clean-text-kobert-tokenized")
dataset

## 토크나이저

In [ ]:
REV = "38279184ba645e8c94d709fbe92eb5bcb47312c1"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True, revision=REV)

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=188, 
        problem_type="multi_label_classification", 
        classifier_dropout=0.5
    )

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self):
        

In [ ]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

In [ ]:
def evaluate_topk(logits, multihot):                        # logits/multihot: [N,188]
    pred_top1 = logits.argmax(axis=1)                       # top-1 예측
    gold_top1 = multihot.argmax(axis=1)                     # 정답 단일화(원본 LabelBinarizer.inverse_transform과 등가)
    out = {
        "weighted_f1": f1_score(gold_top1, pred_top1, average="weighted"),  # baseline headline과 동일
        "micro_f1":    f1_score(gold_top1, pred_top1, average="micro"),
        "macro_f1":    f1_score(gold_top1, pred_top1, average="macro"),
    }
    order = np.argsort(-logits, axis=1)                     # P@k (멀티레이블 참고 지표)
    for k in (1, 3, 5):
        topk = order[:, :k]
        hit = np.take_along_axis(multihot, topk, axis=1).sum(1)
        out[f"p@{k}"] = float((hit / np.clip(multihot.sum(1), 1, k)).mean())
    return out


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return evaluate_topk(np.asarray(logits), np.asarray(labels))

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ration=0.1,
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    num_train_epochs=config["epochs"],
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    logging_dir='./logs',
    logging_steps=50,
    metric_for_best_model="weight_f1",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
trainer.train()